# Preprocessing pipeline -- pilot run (Colab)

A small sample of the real pipeline, to try it before the long run on the server. **The real run is
done from a terminal on the server** (see `docs/preprocessing/README.md`); Colab's disk is wiped when the
session ends, so use this notebook for trying things out only.

Every step below calls the same script you would run on the server: `make_worklist.py` (CT-RATE only) ->
`ingest.py` -> `merge_manifests.py` -> `qc_report.py` -> `assign_splits.py`. Everything one plan produces lives in a
**run** folder, `data/runs/<source>/<run>/` (worklist, manifest, QC report); the preprocessed volumes go to the cache
shared by all runs, `data/cache/` (raw int16 HU, one `.npy` per volume). The manifest carries each volume's kernel,
its labels (when there are any) and the frozen patient-level split.

**Before you start**
- `ctrate`: accept the terms at https://huggingface.co/datasets/ibrahimhamamci/CT-RATE and add a Colab secret named `HF_TOKEN` (key icon, left sidebar; turn on Notebook access).
- `nhrd_local`: put one or two small `.zip` archives of *whole patient folders* (the patient folders directly inside the zip) in a Google Drive folder, and point `ARCHIVE_FOLDER` at it. For a folder shared with you, add a shortcut to My Drive first.

## 1. Upload and extract the project zip

In [ ]:
import shutil
from pathlib import Path

zip_candidates = sorted(Path("/content").glob("*.zip"))
if not zip_candidates:
    from google.colab import files
    print("No zip found in /content yet -- pick your project zip to upload:")
    uploaded = files.upload()
    zip_candidates = [Path("/content") / name for name in uploaded.keys()]

zip_path = zip_candidates[0]
print("Using zip:", zip_path)

In [ ]:
import zipfile

PROJECT_DIR = Path("/content/ChestCT-Multi-Abnormality-Classifier")
extract_dir = Path("/content/_extracted")

for d in (PROJECT_DIR, extract_dir):
    if d.exists():
        shutil.rmtree(d)
extract_dir.mkdir()

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

entries = list(extract_dir.iterdir())
if len(entries) == 1 and entries[0].is_dir():
    shutil.move(str(entries[0]), str(PROJECT_DIR))
else:
    shutil.move(str(extract_dir), str(PROJECT_DIR))

print("Project extracted to:", PROJECT_DIR)

## 2. Install

In [ ]:
%cd /content/ChestCT-Multi-Abnormality-Classifier
!pip install -q ".[dev]"
!python -c "import torch" 2>/dev/null || pip install -q torch

In [ ]:
try:
    import ct_preprocessing  # noqa: F401
    print("ct_preprocessing imported OK -- install succeeded in this runtime")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "ct_preprocessing is not installed in THIS runtime. This is not a bug in the code -- "
        "it happens whenever the Colab runtime restarts or disconnects, which wipes every installed "
        "package even though the notebook still shows old cell output. "
        "Fix: re-run cells 1-2 above (upload/extract the zip, pip install), then run this cell again."
    ) from exc

If a DICOM archive later fails with "pixel data cannot be decoded" (compressed files), set this to `True`, run it, and re-run the ingest step.

In [ ]:
INSTALL_DICOM_CODECS = False #@param {type:"boolean"}
if INSTALL_DICOM_CODECS:
    !pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

## 3. Check the compute available

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "auto" if torch.cuda.is_available() else "cpu"
print("preprocessing device:", DEVICE)

## 4. Choose the source and the pilot size  (the only cell you edit)

Pilot-sized on purpose: a handful of volumes, so it finishes in minutes. `RESET_DATA` starts from an empty `data/` every time.

In [ ]:
SOURCE = "ctrate" #@param ["ctrate", "nhrd_local"]
RESET_DATA = True #@param {type:"boolean"}
CHUNKS_TO_RUN = 3 #@param {type:"integer"}

# --- ctrate: downloads from Hugging Face
TRAIN_KERNEL = "sharp" #@param ["sharp", "soft"]
CHUNK_SIZE = 5 #@param {type:"integer"}
MAX_TRAIN_PATIENTS = 6 #@param {type:"integer"}
MAX_TEST_PATIENTS = 2 #@param {type:"integer"}

# --- nhrd_local: reads the .zip/.tar archives found in this Drive folder
ARCHIVE_FOLDER = "/content/drive/MyDrive/nhrd_raw" #@param {type:"string"}

# --- the split, in patients (ctrate's test set is its own valid pool, so N_TEST_PATIENTS is nhrd_local only)
N_VAL_PATIENTS = 2 #@param {type:"integer"}
N_TEST_PATIENTS = 1 #@param {type:"integer"}

import shlex
if RESET_DATA and Path("data").exists():
    shutil.rmtree("data")
RUN = f"train-{TRAIN_KERNEL}" if SOURCE == "ctrate" else "main"  # the run every step below works on
RUN_DIR = Path("data/runs") / SOURCE / RUN
print("source:", SOURCE, "| run:", RUN, "| device:", DEVICE)

## 5. Get ready to fetch

`ctrate`: logs in to Hugging Face and writes the worklist (downloads only two small metadata CSVs).
`nhrd_local`: mounts Drive and checks the archive folder.

In [ ]:
if SOURCE == "ctrate":
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    cmd = (f"python scripts/preprocessing/make_worklist.py --train-kernel {TRAIN_KERNEL} --chunk-size {CHUNK_SIZE} "
           f"--max-train-patients {MAX_TRAIN_PATIENTS} --max-test-patients {MAX_TEST_PATIENTS}")
    !{cmd}
else:
    if ARCHIVE_FOLDER.startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    folder = Path(ARCHIVE_FOLDER)
    if not folder.is_dir():
        raise FileNotFoundError(f"{ARCHIVE_FOLDER!r} was not found. For a shared folder, add a shortcut to My Drive first.")
    archives = sorted(p.name for p in folder.iterdir() if p.suffix in (".zip", ".tar", ".gz", ".tgz"))
    print(f"{len(archives)} archive(s) found:", archives[:10])

## 6. Ingest

For each chunk: fetch, preprocess into `data/cache/`, record manifest rows, delete the raw data. `--min-free-gb 5` because Colab's disk is small (the server default is 100). Safe to re-run: finished chunks are skipped.

In [ ]:
extra = f" --drive-remote {shlex.quote(ARCHIVE_FOLDER)}" if SOURCE == "nhrd_local" else ""
cmd = (f"python scripts/preprocessing/ingest.py --source {SOURCE} --run {RUN} --max-chunks {CHUNKS_TO_RUN} "
       f"--min-free-gb 5 --workers 1 --device {DEVICE}{extra}")
!{cmd}

In [ ]:
cmd = f"python scripts/preprocessing/ingest.py --source {SOURCE} --run {RUN} --status{extra}"
!{cmd}

## 7. Merge, then quality check

`merge_manifests.py` combines the run's per-chunk manifests into its `manifest.csv` (with kernel columns and, for CT-RATE, labels). `qc_report.py` checks every cached volume and saves montage images to look at.

In [ ]:
!python scripts/preprocessing/merge_manifests.py --source {SOURCE} --run {RUN}
!python scripts/preprocessing/qc_report.py --source {SOURCE} --run {RUN} --montage-every 1

In [ ]:
import pandas as pd
qc = pd.read_csv(RUN_DIR / "qc_report.csv")
print(f"{int(qc.passed.sum())} of {len(qc)} passed")
qc

## 8. Assign the split (by patient, once, then frozen)

Only QC-passed volumes count. Running this again changes nothing for patients that already have a split.

In [ ]:
cmd = f"python scripts/preprocessing/assign_splits.py --source {SOURCE} --run {RUN} --n-val-patients {N_VAL_PATIENTS}"
if SOURCE == "nhrd_local":
    cmd += f" --n-test-patients {N_TEST_PATIENTS}"
!{cmd}

In [ ]:
manifest = pd.read_csv(RUN_DIR / "manifest.csv")
print(manifest.groupby("split").agg(volumes=("volume_id", "count"), patients=("patient_id", "nunique")))
manifest[["volume_id", "patient_id", "kernel", "kernel_class", "split", "qc_passed", "n_slices"]]

## 9. Look at the montages

Should show upright chest slices, lungs dark, soft tissue mid-grey, in the same orientation for every scan.

In [ ]:
from IPython.display import Image, display
montages = sorted(Path("data/qc_montages").glob("*.png"))[:6]
print(f"showing {len(montages)} montage(s)")
for m in montages:
    print(m.name)
    display(Image(filename=str(m)))

## 10. What the model code gets: raw HU, windowed on the fly

The cache is raw int16 HU (one channel). Windowing is the model code's job; this is the agreed 3-channel convention applied to one slice, to show what it looks like. Nothing here is written back to the cache.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

usable = manifest[manifest.split.isin(["train", "val", "test"])]
row = usable.iloc[0]
vol = np.load(f"data/cache/{row.volume_id}.npy", mmap_mode="r")
print(row.volume_id, "->", vol.shape, vol.dtype, "(slices, H, W)")

WINDOWS = {"lung": (-1000, 400), "soft_tissue": (-150, 250), "all_tissue": (-1000, 1000)}
sl = np.asarray(vol[vol.shape[0] // 2])
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(sl, cmap="gray")
axes[0].set_title(f"raw HU {int(sl.min())}..{int(sl.max())}")
for ax, (name, (lo, hi)) in zip(axes[1:], WINDOWS.items()):
    ax.imshow((np.clip(sl, lo, hi) - lo) / (hi - lo), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name} {lo}..{hi}")
for ax in axes:
    ax.axis("off")
plt.show()

## 11. (Optional) Download the results

Colab's disk is wiped when the session ends. This zips the small outputs (manifest, QC report, cache, montages).

In [ ]:
results_dir = Path("/content/pipeline_results")
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir()
shutil.copytree(RUN_DIR, results_dir / "run")  # worklist, manifest, QC report, run record, split copy
for sub in ["cache", "splits"]:
    src = Path("data") / sub
    if src.exists():
        shutil.copytree(src, results_dir / sub)
archive = shutil.make_archive("/content/pipeline_results", "zip", root_dir=str(results_dir))
from google.colab import files
files.download(archive)